In [3]:
import tensorflow as tf
import numpy as np

def process_audio(audio_file_path):
    # Cargar el audio
    audio, sample_rate = tf.audio.decode_wav(tf.io.read_file(audio_file_path), desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)

    # Convertir sample_rate a un valor escalar de Python
    sample_rate = tf.get_static_value(sample_rate)

    # Imprimir la duración del audio
    duration = len(audio) / sample_rate
    print(f"Duración del audio: {duration:.2f} segundos")

    # Parámetros
    window_size_seconds = 1.0
    window_size_samples = int(window_size_seconds * sample_rate)

    # Si el audio es más largo que 1 segundo, encontrar la ventana con mayor energía
    if len(audio) > window_size_samples:
        # Dividir el audio en ventanas superpuestas
        frames = tf.signal.frame(audio, frame_length=window_size_samples, frame_step=window_size_samples // 2)
        
        # Calcular la energía de cada ventana
        energy = tf.reduce_sum(tf.square(frames), axis=1)
        
        # Encontrar el índice de la ventana con mayor energía
        max_energy_index = tf.argmax(energy)
        
        # Seleccionar la ventana con mayor energía
        audio = frames[max_energy_index]
    else:
        # Si el audio es menor a 1 segundo, rellenar con ceros
        padding = tf.zeros([window_size_samples - len(audio)], dtype=audio.dtype)
        audio = tf.concat([audio, padding], 0)
        print("Audio rellenado para alcanzar 1 segundo de duración.")

    # Procesar la ventana seleccionada
    return process_window(audio, sample_rate)

def process_window(window, sample_rate):
    # Parámetros para STFT
    stft_frame_length = int(0.025 * sample_rate)  # 25ms
    stft_frame_step = int(0.010 * sample_rate)    # 10ms

    # STFT
    stft = tf.signal.stft(window,
                          frame_length=stft_frame_length,
                          frame_step=stft_frame_step,
                          fft_length=2048)

    # Magnitud del espectrograma
    spectrogram = tf.abs(stft)

    # Convertir a mel spectrogram
    num_spectrogram_bins = spectrogram.shape[-1]
    lower_edge_hertz, upper_edge_hertz, num_mel_bins = 20.0, sample_rate / 2, 96
    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins, num_spectrogram_bins, sample_rate, lower_edge_hertz, upper_edge_hertz)

    mel_spectrogram = tf.tensordot(spectrogram, linear_to_mel_weight_matrix, 1)
    mel_spectrogram.set_shape(spectrogram.shape[:-1].concatenate(linear_to_mel_weight_matrix.shape[-1:]))

    # Aplicar transformación logarítmica
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)

    # Calculo los MFCCs a partir del log_mel_spectrograms y tomo los primeros 13
    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel_spectrogram)[..., :13]

    return mfccs, log_mel_spectrogram


In [4]:
# Ejemplo de uso
audio_file = './0abd0eb8.wav'
mfccs, log_mel_spectrograms = process_audio(audio_file)

print(f"Forma de los MFCCs: {mfccs.shape}")
print(f"Forma de los espectrogramas log-mel: {log_mel_spectrograms.shape}")

Duración del audio: 0.48 segundos
Audio rellenado para alcanzar 1 segundo de duración.
Forma de los MFCCs: (98, 13)
Forma de los espectrogramas log-mel: (98, 96)
